# Agents War — starter notebook

**Google Colab:** open this notebook from the GitHub repo in Colab
(File → Open notebook → GitHub tab → paste the repo link), then run the cells top to bottom (Shift + Enter).
The first cell downloads the rest of the kit for you.

**Laptop:** `git clone` the repo (or Code → Download ZIP), open this notebook inside that folder and run it.

⚠️ Colab deletes your files when the session ends. Run the **last cell** often to download `strategy.py`.
That file is what you submit.

In [ ]:
# ---- Setup (run once) ----
REPO_URL = "https://github.com/0770-Lakshya/ALPHA-ARENA-Agents-War-.git"
import os, subprocess, importlib
if not os.path.exists("engine.py"):
    folder = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if not os.path.isdir(folder):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir(folder)

import engine
data = engine.load("data/train.csv")
print("Kit ready:", len(data["close"]), "days x", data["close"].shape[1], "stocks")
data["close"].tail()

In [ ]:
# Look at the stocks (all start at 100)
data["close"].plot(figsize=(12, 5), legend=False, title="Training prices");

## Your strategies
**This cell is your `strategy.py`.** Edit it and run it: the first line saves it to the file.
Write each idea as its own `decide_xxx` function and point `decide` at your final one.
(Use either this notebook or edit `strategy.py` directly, not both: running this cell overwrites the file.)

In [ ]:
%%writefile strategy.py
"""
AGENTS WAR - your trading agent.  This is the ONLY code file you submit.

How to work:
  1. Write every idea you try as its own function named  decide_<something>(history).
  2. Run  python compare.py  to test all of them side by side (table + chart for your slides).
  3. Point  decide  (bottom of this file) at the one you want us to run on the secret period.

Rules for this file:
  * `decide` must exist - that is the function we run.
  * Allowed imports: python standard library, numpy, pandas, scikit-learn.
  * No internet, no reading other files, no hard-coded prices or day numbers
    (your agent must not behave differently just because of which day number it is).
  * On average your agent must keep at least 30% of the money invested.
  * A full run (training + secret days) must finish within 3 minutes.
"""
import numpy as np
import pandas as pd

TEAM_NAME = "<Your team name here>"

# history: dict with keys "open", "high", "low", "close", "volume".
#          Each one is a pandas DataFrame: rows = days so far (last row = today),
#          columns = stocks (STOCK_01 ... STOCK_12).
# return:  dict {stock: weight}
#          weight =  0.25 -> put 25% of your money in that stock (buy)
#          weight = -0.10 -> bet 10% against that stock (short sell)
#          stocks you leave out get 0; whatever is unused stays in cash.
#          The sum of |weights| must be <= 1 (scaled down for you if it is more).


def decide_momentum(history):
    """Buy the 3 stocks that went up the most in the last 20 days."""
    close = history["close"]
    if len(close) < 21:
        return {}                                   # not enough data yet -> stay in cash
    past_return = close.iloc[-1] / close.iloc[-21] - 1
    winners = past_return.nlargest(3).index
    return {stock: 1 / 3 for stock in winners}


def decide_trend_filter(history):
    """Hold every stock that is above its 50-day average, equally weighted."""
    close = history["close"]
    if len(close) < 50:
        return {}
    above = close.iloc[-1] > close.iloc[-50:].mean()
    picks = above[above].index
    return {stock: 1 / len(picks) for stock in picks} if len(picks) else {}


# Ideas to try next: mean reversion, RSI, volatility filters, stop-losses,
# position sizing by risk, ML models (train once, cache in a global), ...


decide = decide_trend_filter      # <- the agent we will run on the secret period

## Compare all your strategies
Splits the data into **fit** (first 10 years: tune only here) and **validation** (the years after).
It saves `comparison.csv` and `comparison.png` for your submission and slides.

In [ ]:
import strategy, compare
importlib.reload(strategy); importlib.reload(compare)
compare.main()

## Your chosen agent (`decide`) over all training days

In [ ]:
importlib.reload(strategy)
res = engine.backtest(strategy.decide, data)
bench = engine.backtest(engine.equal_weight, data, verbose=False)
engine.plot(res, f"{strategy.TEAM_NAME} - {strategy.decide.__name__}", benchmark=bench)

In [ ]:
# What did the agent hold on the last 10 days?
res["weights"].tail(10).round(2)

## Download your submission files

In [ ]:
try:
    from google.colab import files
    files.download("strategy.py")
    if os.path.exists("comparison.csv"):
        files.download("comparison.csv")
except ImportError:
    print("Not on Colab - your files are in:", os.getcwd())